# 17 · Production = M2.0: the checks (branch `feat/model_v3_m2_0`)

## Executive summary (read this first)

On branch `feat/model_v3_m2_0`, `forecast_models.py` (what the container runs) is **M2.0**, the model of notebook 15.
This notebook checks that the two are the same model, number for number. Notebook 11 did the same for M1.5; it re-runs on
`feat/model_v3_m1_5`, where production is still M1.5.

**The checks:**
1. **Draws at the as-of.** With the text neutral and seed = crc32(unit id), `forecast_models.build_draws` with 2000 draws
   returns notebook 15's draws on all 103 units. The largest difference is printed; 0 means identical.
2. **Draws on the backtest.** The same on every 5th backtest date of the 99 daily units. The 4 monthly units are checked at
   the as-of only: on past dates the notebooks use M0's fixed month table, production counts months from the dates.
3. **The switches.** Production's module switches (RECENT_TREND, ZERO_FLOOR, VR_SHRINK, TREND_SHARE, VOL_POOL) turn the pieces off one by one. Each combination is
   checked against the notebook's model with the same pieces off, down to M1.5, at every card's as-of.

Production reads only the unit folder: each asset's type comes from the panel that holds it, as notebook 11 shows. No
realized outcome is read.

## 0 — Setup (as notebook 07), plus production

In [1]:
import bisect, multiprocessing as mp, pathlib, sys, time, tomllib, warnings, zlib
import numpy as np, pandas as pd, pyarrow.parquet as pq

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "model_experiment_v3" else pathlib.Path.cwd() / "model_experiment_v3"
REPO, UNITS, DATA = HERE.parent, HERE.parent / "units", HERE / "data"
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
import m0_baseline as m0
from qfbench2_track_forecasting import scoring as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))

ANSWERS = pd.read_parquet(DATA / "backtest_answers.parquet")
M0_SCORES = pd.read_parquet(DATA / "m0_backtest_scores.parquet")
M11_SCORES = pd.read_parquet(DATA / "m1_1_backtest_scores.parquet")
UNIT_IDS = sorted(ANSWERS.unit.unique())
RULES = tomllib.loads((HERE / "transformations.toml").read_text())
SH = RULES["shocks"]
WINDOW = 260

def asset_type(a):
    return RULES["assets"].get(a, "other")

def rule_for(asset, target_type):
    """card.toml's target type + the asset -> {transform, shock, drift} (transformations.toml)."""
    t = RULES["assets"].get(asset)
    base = RULES["types"][t] if t else {"transform": RULES["default"]["level"], "shock": RULES["default"]["shock"],
                                        "drift": RULES["default"]["drift"]}
    tr = RULES["default"]["log_return"] if target_type == "log_return" else base["transform"]
    return {"transform": tr, "shock": base["shock"], "drift": bool(base["drift"])}

def load_unit(u):
    ud = UNITS / u
    card = tomllib.loads((ud / "card.toml").read_text()); t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    uid, asof, tt = card["task"]["id"], card["provenance"]["data_cutoff"], t.get("target_type", "level")
    rows = {}
    for a in assets:
        for path in sorted(ud.glob("*.parquet")):
            tab = pq.read_table(path).to_pydict()
            acol = "asset" if "asset" in tab else ("asset_id" if "asset_id" in tab else None)
            if acol is None:
                continue
            r = sorted((str(d)[:10], float(v)) for d, x, v in zip(tab["date"], tab[acol], tab["value"])
                       if str(x) == a and str(d)[:10] <= asof and v is not None)
            if r:
                rows[a] = r; break
    return {"unit": u, "id": uid, "family": card["metadata"]["category"], "card": card, "assets": assets,
            "horizons": horizons, "target_type": tt, "asof": asof, "rows": rows, "dates": {a: [d for d, _ in rows[a]] for a in assets},
            "rules": {a: rule_for(a, tt) for a in assets}, "steps": {h: m0._MONTHLY_STEPS.get(uid, {}).get(h, h) for h in horizons}}

UNITS_DATA = {u: load_unit(u) for u in UNIT_IDS}
TYPE = {u: "+".join(sorted({asset_type(a) for a in U["assets"]})) for u, U in UNITS_DATA.items()}
BACKTEST = {}
for u, g in ANSWERS.groupby("unit"):
    U = UNITS_DATA[u]
    order = {(a, h): i for i, (a, h) in enumerate((a, h) for a in U["assets"] for h in U["horizons"])}
    BACKTEST[u] = [(o, gg.assign(i=[order[(a, h)] for a, h in zip(gg.asset, gg.horizon)]).sort_values("i").answer.to_numpy())
                   for o, gg in g.groupby("origin")]
print(f"{len(UNIT_IDS)} units, {sum(len(v) for v in BACKTEST.values()):,} backtest dates")
display(pd.DataFrame({t: {"transform": v["transform"], "shock": v["shock"], "drift": v["drift"]} for t, v in RULES["types"].items()}).T)
print("Student-t:", SH)

import zlib
from unittest import mock
import forecast_agent as fa, forecast_models as fm
NEUTRAL = {"shift": 0.0, "widen": 1.0, "skew": 0.0}
PANELS = {u: fa._read_panels(UNITS / u) for u in UNIT_IDS}
SEED = {u: zlib.crc32(UNITS_DATA[u]["id"].encode()) & 0x7FFFFFFF for u in UNIT_IDS}
MONTHLY = [u for u in UNIT_IDS if UNITS_DATA[u]["id"] in m0._MONTHLY_STEPS]
N_DRAWS = 2000
def production(u, origin):
    U = UNITS_DATA[u]
    return fm.build_draws(PANELS[u], U["assets"], U["horizons"], origin, {a: dict(NEUTRAL) for a in U["assets"]},
                          N_DRAWS, SEED[u], target_type=U["target_type"], family=U["family"])
print("production switches:", {k: getattr(fm, k) for k in "RECENT_TREND, ZERO_FLOOR, VR_SHRINK, TREND_SHARE, VOL_POOL".split(", ")})

103 units, 23,635 backtest dates


,transform,shock,drift
rates,diff,student_t,False
fx,log_diff,student_t,False
cpi,log_diff,normal,True
payrolls,diff,normal,True
unemployment,diff,normal,False
equity_factor,as_return,student_t,False


Student-t: {'nu': 'fit', 'nu_min': 4.2, 'nu_max': 30.0, 'normal_below_kurtosis': 0.2}
production switches: {'RECENT_TREND': True, 'ZERO_FLOOR': True, 'VR_SHRINK': 30.0, 'TREND_SHARE': 0.25, 'VOL_POOL': (21, 63, 252)}


## 1 — Notebook 15's model (M2.0), unchanged

In [2]:
def fit_nu(x, rows_t):
    """nu from the window's excess kurtosis k: a Student-t has k = 6 / (nu - 4), so nu = 4 + 6 / k."""
    if not rows_t:
        return None
    z = x[rows_t] - x[rows_t].mean(axis=1, keepdims=True)
    k = float(np.mean((z ** 4).mean(axis=1) / (z ** 2).mean(axis=1) ** 2 - 3))
    return None if k < SH["normal_below_kurtosis"] else float(np.clip(4 + 6 / k, SH["nu_min"], SH["nu_max"]))

def fit(U, origin, steps_window=None):
    """The window's steps per asset -> (assets sorted, steps matrix x, log mode, last value)."""
    by_asset = sorted(U["assets"]); steps, log_mode, last = {}, {}, {}
    for a in by_asset:
        k = bisect.bisect_right(U["dates"][a], origin)
        rows = U["rows"][a][max(0, k - (m0._WINDOW if steps_window is None else steps_window + 1)): k]
        log_mode[a] = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
        if log_mode[a]:
            steps[a], _ = m0._steps([(d, np.log(v)) for d, v in rows], "level"); last[a] = rows[-1][1]
        else:
            steps[a], lv = m0._steps(rows, U["target_type"]); last[a] = lv if U["target_type"] == "level" else 0.0
    common = sorted(set.intersection(*(set(steps[a]) for a in by_asset)))
    x = np.array([[steps[a][d] for d in common] for a in by_asset])
    return by_asset, x, log_mode, last

def ewma_sd_asset(U, a, origin, halflife):
    """EWMA sd of one asset's steps (its own transform) over the last 8 x halflife steps: weight 0.5 ** (age / halflife)."""
    k = bisect.bisect_right(U["dates"][a], origin)
    rows = U["rows"][a][max(0, k - (int(8 * halflife) + 1)): k]
    lm = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
    st, _ = m0._steps([(d, np.log(v)) for d, v in rows] if lm else rows, "level" if lm else U["target_type"])
    v = np.array([st[d] for d in sorted(st)])
    age = np.arange(len(v))[::-1]; w = 0.5 ** (age / halflife); w = w / w.sum()
    m = w @ v
    return float(np.sqrt(w @ (v - m) ** 2))

# Every asset's whole history as steps (its own transform, M0's gap rule), computed once: the recent trend (M1.7) looks
# back 126 steps, the variance ratio (M1.8) over the whole history.
FULL = {}
for _u, _U in UNITS_DATA.items():
    for _a in _U["assets"]:
        _rows = _U["rows"][_a]
        _lm = _U["rules"][_a]["transform"] == "log_diff" and min(v for _, v in _rows) > 0
        _st, _ = m0._steps([(d, np.log(v)) for d, v in _rows] if _lm else _rows, "level" if _lm else _U["target_type"])
        _ds = sorted(_st)
        FULL[(_u, _a)] = (np.array([str(d) for d in _ds]), np.array([_st[d] for d in _ds]))

def hist_steps(U, a, origin):
    """All of one asset's steps up to the origin, oldest first."""
    d, v = FULL[(U["unit"], a)]
    return v[: np.searchsorted(d, origin, side="right")]

def recent_trend(U, a, origin, mu_window):
    """M1.7's drift per step.
    CPI, NFP (drift by type): the mean of the last 12 monthly steps -- this year's trend, not the 21-year average.
    Everything else: the mean m of the last 126 steps, shrunk toward 0 by its t-statistic t = m / (sd / sqrt(126))
    (James-Stein): drift = m x max(0, 1 - 4 / t^2). |t| <= 2 gives no drift at all; |t| = 4 keeps 75 %."""
    v = hist_steps(U, a, origin)
    if U["rules"][a]["drift"]:
        return float(v[-12:].mean()) if len(v) >= 12 else mu_window
    v = v[-126:]
    sd = v.std(ddof=1)
    t = v.mean() / (sd / np.sqrt(len(v))) if sd > 0 else 0.0
    return float(v.mean() * max(0.0, 1.0 - 4.0 / max(t * t, 1e-12)))

def variance_ratio(U, a, h_steps, origin, n0=30.0):
    """M1.8: VR = mean(h-step move^2) / (h x mean(1-step move^2)) over the asset's whole history before the origin.
    VR > 1: moves add up faster than sqrt(h) (trends, cycles); VR < 1: they partly cancel (mean reversion). Raw moves (no
    demeaning) for assets without a drift, because a trend we do not forecast is width. Shrunk toward 1 by the number of
    independent windows: VR = 1 + (VR_raw - 1) x n / (n + 30), n = history steps / h."""
    v = hist_steps(U, a, origin)
    if h_steps <= 1 or len(v) < 2 * h_steps + 20:
        return 1.0
    m = v.mean() if U["rules"][a]["drift"] else 0.0
    c = np.concatenate([[0.0], np.cumsum(v - m)])
    ratio = np.mean((c[h_steps:] - c[:-h_steps]) ** 2) / (h_steps * np.mean((v - m) ** 2))
    n_eff = len(v) / h_steps
    return 1.0 + (ratio - 1.0) * n_eff / (n_eff + n0)

def m20_draws(U, origin, drift17=True, floor=True, vr=True, trend_mix=0.25, vol_pool=(21, 63, 252), n_draws=2000, ewma_halflife=63):
    """M2.0 = M1.9 + an ensemble of volatility estimates (vol_pool: the draws take turns over EWMA halflives 21, 63 and
    252) + 2000 draws. vol_pool=None, n_draws=500 is M1.9 (notebook 14)."""
    N = n_draws
    by_asset, x, log_mode, last = fit(U, origin, WINDOW)
    assets, horizons = U["assets"], U["horizons"]
    mu, sigma = x.mean(axis=1), np.atleast_2d(np.cov(x))
    sd_w = np.sqrt(np.diag(sigma)); corr = np.nan_to_num(sigma / np.outer(sd_w, sd_w), nan=0.0); np.fill_diagonal(corr, 1.0)
    sd_e = np.array([ewma_sd_asset(U, a, origin, ewma_halflife) for a in by_asset])     # M1.5: EWMA sd, halflife 63
    sigma = corr * np.outer(sd_e, sd_e)
    ai = {a: i for i, a in enumerate(by_asset)}
    cells = [(a, h) for a in by_asset for h in sorted(horizons)]; s = U["steps"]
    drift = {a: (mu[ai[a]] if U["rules"][a]["drift"] else 0.0) for a in by_asset}      # M1.5: drift by type
    if drift17:                                                                          # M1.7: the recent trend
        drift = {a: recent_trend(U, a, origin, mu[ai[a]]) for a in by_asset}
    def centre(a, h, d):
        return (0.0 if log_mode[a] else last[a]) + s[h] * d[a]
    mean = np.array([centre(a, h, drift) for a, h in cells])
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10; j = cov + 1e-9 * np.eye(len(cells))
    try: f = np.linalg.cholesky(j)
    except np.linalg.LinAlgError: f = np.diag(np.sqrt(np.diag(j)))
    seed = zlib.crc32(U["id"].encode()) & 0x7FFFFFFF
    dev = np.random.default_rng(seed).standard_normal((N, len(cells))) @ f.T
    scale = np.ones(len(cells))
    if vr:
        scale = scale * np.array([np.sqrt(variance_ratio(U, a, s[h], origin)) for a, h in cells])
    dev = dev * scale
    if vol_pool:                        # M2.0: draw i takes the EWMA sd of halflife vol_pool[i % 3]
        ratio = np.array([[ewma_sd_asset(U, a, origin, hl) / sd_e[ai[a]] if sd_e[ai[a]] > 0 else 1.0
                           for a, h in cells] for hl in vol_pool])
        dev = dev * ratio[np.arange(N) % len(vol_pool)]
    t_cells = np.array([U["rules"][a]["shock"] == "student_t" for a, h in cells])
    nu = fit_nu(x, [ai[a] for a in by_asset if U["rules"][a]["shock"] == "student_t"])
    if nu is not None and t_cells.any():
        dev = dev * np.where(t_cells, np.sqrt((nu - 2.0) / np.random.default_rng([seed, 7]).chisquare(nu, size=(N, 1))), 1.0)
    sm = mean + dev
    if trend_mix:
        k = int(round(trend_mix * N))           # the last k draws: the full trend of the window
        full = {a: (drift[a] if U["rules"][a]["drift"] else mu[ai[a]]) for a in by_asset}
        sm[N - k:] = np.array([centre(a, h, full) for a, h in cells]) + dev[N - k:]
    out = np.empty((N, len(assets), len(horizons)))
    for ci, (a, h) in enumerate(cells):
        col = last[a] * np.exp(sm[:, ci]) if log_mode[a] else sm[:, ci]
        if floor and asset_type(a) == "rates":
            col = np.maximum(col, 0.0)          # zero lower bound on Treasury yields
        if s[h] <= 0:
            col = np.full(N, U["rows"][a][bisect.bisect_right(U["dates"][a], origin) - 1][1])
        out[:, assets.index(a), horizons.index(h)] = col
    return out

## 2 — Checks 1 and 2: the draws

In [3]:
asof_diff = pd.Series({u: float(np.max(np.abs(production(u, UNITS_DATA[u]["asof"]) - m20_draws(UNITS_DATA[u], UNITS_DATA[u]["asof"]))))
                      for u in UNIT_IDS})
print(f"check 1 — at the as-of, 103 units: largest |production - notebook 15| = {asof_diff.max():.1e} "
      f"(units with any difference: {(asof_diff > 0).sum()})")
assert asof_diff.max() == 0.0

def backtest_diff(u):
    U = UNITS_DATA[u]; dates = BACKTEST[u][::5]
    return u, len(dates), max(float(np.max(np.abs(production(u, o) - m20_draws(U, o)))) for o, _ in dates)
t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    BT = pd.DataFrame(pool.map(backtest_diff, [u for u in UNIT_IDS if u not in MONTHLY], chunksize=1),
                      columns=["unit", "dates", "largest difference"]).set_index("unit")
print(f"check 2 — every 5th backtest date of the {len(BT)} daily units ({BT.dates.sum():,} dates, {time.time() - t0:.0f}s): "
      f"largest |production - notebook 15| = {BT['largest difference'].max():.1e}")
assert BT["largest difference"].max() == 0.0

check 1 — at the as-of, 103 units: largest |production - notebook 15| = 0.0e+00 (units with any difference: 0)


check 2 — every 5th backtest date of the 99 daily units (4,758 dates, 35s): largest |production - notebook 15| = 0.0e+00


## 3 — Check 3: each switch gives back the previous model

In [4]:
rows = {}
for label, prod_off, nb_off in [("VOL_POOL = None is M1.9 (with these draws)", dict(VOL_POOL=None), dict(vol_pool=None)),
         ("+ TREND_SHARE = 0 is M1.8", dict(VOL_POOL=None, TREND_SHARE=0.0), dict(vol_pool=None, trend_mix=0.0)),
         ("+ VR_SHRINK = None, ZERO_FLOOR = False is M1.7", dict(VOL_POOL=None, TREND_SHARE=0.0, VR_SHRINK=None, ZERO_FLOOR=False), dict(vol_pool=None, trend_mix=0.0, vr=False, floor=False)),
         ("+ RECENT_TREND = False is M1.5", dict(VOL_POOL=None, TREND_SHARE=0.0, VR_SHRINK=None, ZERO_FLOOR=False, RECENT_TREND=False), dict(vol_pool=None, trend_mix=0.0, vr=False, floor=False, drift17=False))]:
    with mock.patch.multiple(fm, **prod_off):
        d = max(float(np.max(np.abs(production(u, UNITS_DATA[u]["asof"]) - m20_draws(UNITS_DATA[u], UNITS_DATA[u]["asof"], **nb_off))))
                for u in UNIT_IDS)
    rows[label] = {"production switches": ", ".join(f"{k}={v}" for k, v in prod_off.items()), "largest difference (103 units)": d}
    assert d == 0.0
pd.DataFrame(rows).T

,production switches,largest difference (103 units)
VOL_POOL = None is M1.9 (with these draws),VOL_POOL=None,0
+ TREND_SHARE = 0 is M1.8,"VOL_POOL=None, TREND_SHARE=0.0",0
"+ VR_SHRINK = None, ZERO_FLOOR = False is M1.7","VOL_POOL=None, TREND_SHARE=0.0, VR_SHRINK=None...",0
+ RECENT_TREND = False is M1.5,"VOL_POOL=None, TREND_SHARE=0.0, VR_SHRINK=None...",0
